# 🏢 Hierarchical Agents in LangGraph

## Learning Objectives
In this notebook, you will learn:
1. **Subgraphs as departments** - compile independent LangGraph graphs and plug each one in as a single node of a parent graph.
2. **Shared state contracts** - design one `TeamState` schema that every department and the parent speak, and extend it (`CEOState`, `ContentState`) with keys only one level uses.
3. **Fan-out / fan-in patterns** - run multiple specialists in parallel inside a department, then merge their outputs with a lead node.
4. **A department with its own supervisor** - the content team's supervisor routes between a writer and an editor with the same `RouteDecision` type as `01_supervisor_pattern.ipynb`, capped by a counter reducer.
5. **A CEO that chains departments** - control returns to the CEO after every department, so it can send the work on to a second department or finish; it routes on the typed decision stored in state, capped by a counter reducer.
6. **Tracing multi-level execution** - follow one request from the CEO down to a department's supervisor and specialists, and back up.

## Where this fits
`01_supervisor_pattern.ipynb` built one supervisor over three specialists. This notebook stacks supervisors: a CEO
supervisor sends work to department subgraphs and takes control back after each one, and the content department runs
its own supervisor. Module 05's `03_five_multi_agent_architectures.ipynb` (Part 4) sketched the same shape with
hard-coded routing; here both levels decide with an LLM.

## Prerequisites
- Familiarity with core LangGraph concepts: `StateGraph`, `START`/`END`, conditional edges (see [repository 04 — LangGraph fundamentals](https://github.com/Sourav692/ai-engineering-demystified-04-langgraph-fundamentals)).
- `01_supervisor_pattern.ipynb` (single-level supervisor with typed routing, a route log and a counter-reducer cap).
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env`
- Basic understanding of Pydantic models for structured LLM output.

---
## 📦 Step 1: Environment Setup

The setup cell loads `OPENAI_API_KEY` and sets the two loop caps; the next cell creates the one LLM shared by the CEO, the content supervisor, every department lead and every specialist.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import operator
import warnings

from typing import Literal

from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from pydantic import BaseModel, Field
from typing_extensions import Annotated, TypedDict
from dotenv import find_dotenv, load_dotenv

# --- Loop caps --------------------------------------------------------------
# Both supervisors in this notebook loop: their workers hand control back to them.
# Each cap is enforced by a pure routing function that reads a counter in state.
MAX_CEO_TURNS = 4  # CEO decisions per request: at most 3 department runs, then stop
MAX_CONTENT_TURNS = 4  # content-supervisor decisions per visit: writer, editor, FINISH fit in 3

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: the CEO, the content supervisor, department leads and specialists
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini", temperature=0)

print(f"✅ LLM initialized: {llm.model_name}")

---
## 🗂️ Part 1: Shared State Schema

Every level of the hierarchy - the parent graph and each department subgraph - speaks the *same* contract, `TeamState`. That is what makes a compiled subgraph pluggable as a single node in the parent graph: the parent passes its state in, the department reads the `TeamState` keys, and its result flows back through the same keys.

A level may extend the contract with keys only it uses. The parent's `CEOState` adds the CEO's routing decision, a route log and a turn counter; the content team's `ContentState` (Part 2.2) adds its supervisor's decision and counter. Keys outside `TeamState` never cross the boundary: a department ignores the parent's extra keys, and the parent drops a department's.

### Key Concepts:
- **`messages`**: the shared work transcript, using LangGraph's `add_messages` reducer so new messages are appended rather than overwriting the list.
- **`final_answer`**: a scratch field each department's "lead" node writes to, so the parent graph can read a single clean output regardless of which department handled the request.
- **`department`** (`CEOState`): the CEO's typed routing decision. The routing function reads this field — never the CEO's free-text reasoning.
- **`route_log`** and **`ceo_turns`** (`CEOState`): one entry per CEO decision (kept out of `messages`, as in `01_supervisor_pattern.ipynb`) and the counter the CEO's cap reads.

> **Key Insight**: keep counters and logs out of the shared contract. A department hands its whole final state back to the parent, and the parent applies its reducers to it. `add_messages` merges messages by id, so the returned transcript adds nothing twice; an `operator.add` counter in `TeamState` would be added a second time.

In [ ]:
# ============================================================================
# TEAM STATE: the contract every level shares, plus the CEO's own keys
# ============================================================================
class TeamState(TypedDict):
    """The contract every department subgraph and the parent graph share."""

    messages: Annotated[list[BaseMessage], add_messages]
    final_answer: str


class CEOState(TeamState):
    """The parent graph's state: TeamState plus keys only the CEO uses."""

    department: str  # the CEO's typed routing decision, written by the ceo node
    route_log: Annotated[list[dict], operator.add]  # one entry per CEO decision (never in messages)
    ceo_turns: Annotated[int, operator.add]  # counter reducer: +1 per CEO decision


print(f"✅ TeamState: {list(TeamState.__annotations__)}")
print(f"✅ CEOState adds: {[k for k in CEOState.__annotations__ if k not in TeamState.__annotations__]}")

---
## 🏛️ Part 2: Department Subgraphs

Each department below is built as its own independent `StateGraph`, compiled on its own, and later attached to the parent graph as a single node. This is the core idea of hierarchical agent systems: complexity is hidden inside subgraphs, so the top-level graph only has to reason about "which department" rather than "which of N specialists". The research and analysis teams are fixed pipelines; the content team has its own supervisor (2.2).

### 2.1 🔬 Research Team

The research department fans out to two specialists that run independently - a "web researcher" and a paper reviewer, both answering
from the model's own knowledge (no search tool here, to keep the focus on the structure) - then fans back in to a research lead that synthesizes both findings into a single brief.

In [ ]:
# ============================================================================
# BUILD_RESEARCH_TEAM: Research department: web + paper specialists -> lead
# ============================================================================
def build_research_team() -> StateGraph:
    """Build the research department subgraph."""

    def web_researcher(state: TeamState) -> dict:
        """Searches the web for information."""
        # Extract the query from the last human message
        query = ""
        for msg in reversed(state["messages"]):
            if isinstance(msg, HumanMessage):
                query = msg.content
                break

        response = llm.invoke(
            [
                SystemMessage(
                    content=(
                        "You are a web researcher. Find key facts and data about "
                        "the topic. Provide 3-4 bullet points of findings. Be specific."
                    )
                ),
                HumanMessage(content=query),
            ]
        )
        return {
            "messages": [
                AIMessage(
                    content=f"[WEB RESEARCHER]: {response.content}",
                    name="web_researcher",
                )
            ]
        }

    def paper_reviewer(state: TeamState) -> dict:
        """Reviews academic/technical sources."""
        query = ""
        for msg in reversed(state["messages"]):
            if isinstance(msg, HumanMessage):
                query = msg.content
                break

        response = llm.invoke(
            [
                SystemMessage(
                    content=(
                        "You are an academic reviewer. Provide technical depth and "
                        "cite relevant concepts or frameworks. 3-4 bullet points."
                    )
                ),
                HumanMessage(content=query),
            ]
        )
        return {
            "messages": [
                AIMessage(
                    content=f"[PAPER REVIEWER]: {response.content}",
                    name="paper_reviewer",
                )
            ]
        }

    def research_lead(state: TeamState) -> dict:
        """Synthesizes findings from both researchers."""
        response = llm.invoke(
            [
                SystemMessage(
                    content=(
                        "You are the research lead. Synthesize the web researcher's "
                        "and paper reviewer's findings into a cohesive research brief. "
                        "Keep it to one short paragraph."
                    )
                ),
                *state["messages"],
            ]
        )
        return {
            "messages": [
                AIMessage(
                    content=f"[RESEARCH LEAD]: {response.content}", name="research_lead"
                )
            ],
            "final_answer": response.content,
        }

    # Build the research subgraph
    research_graph = StateGraph(TeamState)
    research_graph.add_node("web_researcher", web_researcher)
    research_graph.add_node("paper_reviewer", paper_reviewer)
    research_graph.add_node("research_lead", research_lead)

    # Fan-out: both researchers work in parallel
    research_graph.add_edge(START, "web_researcher")
    research_graph.add_edge(START, "paper_reviewer")

    # Fan-in: both feed into the research lead
    research_graph.add_edge("web_researcher", "research_lead")
    research_graph.add_edge("paper_reviewer", "research_lead")
    research_graph.add_edge("research_lead", END)

    return research_graph

#### 🧪 Try it: Single Department in Isolation

Before wiring departments into the full hierarchy, it helps to run one department subgraph on its own to confirm the fan-out/fan-in wiring behaves as expected.

In [ ]:
# ============================================================================
# DEMO_SINGLE_DEPARTMENT: Run the research team subgraph standalone
# ============================================================================
def demo_single_department():
    """Demo a single department subgraph in isolation."""
    print("Single Department Demo (Research Team):\n")

    research_team = build_research_team().compile()
    result = research_team.invoke(
        {
            "messages": [
                HumanMessage(content="What is retrieval-augmented generation (RAG)?")
            ],
            "final_answer": "",
        }
    )

    for msg in result["messages"]:
        if isinstance(msg, AIMessage):
            print(f"{msg.content[:200]}...\n")

    print(f"Research Brief:\n{result['final_answer']}")

In [ ]:
# ============================================================================
# TRY IT: THE RESEARCH TEAM ON ITS OWN
# ============================================================================
demo_single_department()

### 2.2 ✍️ Content Team (with its own supervisor)

The content department has a writer, who drafts a piece from whatever context is already in the conversation (e.g. a research brief), and an editor, who polishes the latest draft into the final answer. A **content supervisor** decides who works next: it returns the same `RouteDecision` type as `01_supervisor_pattern.ipynb` (a closed `Literal` of this team's members plus `FINISH`, and a reason), and every specialist hands control back to it.

It is built like the supervisor in `01_supervisor_pattern.ipynb`:
- it reads a short status built in code (drafts written, drafts polished) and follows explicit routing rules;
- it writes its decision to state (`next_step`), and a pure function, `route_content`, routes on it;
- a counter reducer (`content_turns`) caps it at `MAX_CONTENT_TURNS` decisions per visit.

`ContentState` extends `TeamState` with those two keys. They stay inside the department: the parent graph never sees them. The supervisor also prints each decision (indented), so a run of the full hierarchy shows both levels deciding.

In [ ]:
# ============================================================================
# BUILD_CONTENT_TEAM: Content department: a supervisor over a writer and an editor
# ============================================================================
class RouteDecision(BaseModel):
    """A supervisor's routing decision: the type from 01_supervisor_pattern.ipynb, with this team's members."""

    next: Literal["writer", "editor", "FINISH"] = Field(
        description="Which specialist acts next, or FINISH when the latest draft is polished"
    )
    reasoning: str = Field(description="Why this specialist was chosen")


class ContentState(TeamState):
    """TeamState plus the content supervisor's own keys (they never reach the parent)."""

    next_step: str  # the content supervisor's typed decision
    content_turns: Annotated[int, operator.add]  # counter reducer: +1 per supervisor decision


CONTENT_SUPERVISOR_PROMPT = """You supervise a content team:
- writer: drafts the piece from the request and any research in the conversation
- editor: polishes the latest draft into the final version

You get a status line with two counts. Follow these rules exactly:
1. Drafts written is 0 -> writer
2. Drafts polished is lower than drafts written -> editor
3. Drafts polished equals drafts written (and is at least 1) -> FINISH"""


def build_content_team() -> StateGraph:
    """Build the content department subgraph: a supervisor over a writer and an editor."""
    supervisor_llm = llm.with_structured_output(RouteDecision)

    def content_supervisor(state: ContentState) -> dict:
        """LLM decision, written to state; route_content does the routing."""
        names = [m.name for m in state["messages"] if isinstance(m, AIMessage)]
        status = (
            f"Drafts written: {names.count('content_writer')}. "
            f"Drafts polished: {names.count('content_editor')}."
        )
        decision = supervisor_llm.invoke(
            [SystemMessage(content=CONTENT_SUPERVISOR_PROMPT), HumanMessage(content=status)]
        )
        print(f"   📝 content supervisor → {decision.next}: {decision.reasoning}")
        return {"next_step": decision.next, "content_turns": 1}

    def route_content(state: ContentState) -> str:
        """Pure routing on the typed decision, never past the department's cap."""
        if state["content_turns"] >= MAX_CONTENT_TURNS:
            print(f"   ⛔ content supervisor cap ({MAX_CONTENT_TURNS}) reached")
            return END
        return END if state["next_step"] == "FINISH" else state["next_step"]

    def content_writer(state: TeamState) -> dict:
        """Writes content based on available context."""
        response = llm.invoke(
            [
                SystemMessage(
                    content=(
                        "You are a skilled content writer. Using any research or context "
                        "in the conversation, write a clear, engaging short piece "
                        "(one paragraph). Match a professional but accessible tone."
                    )
                ),
                *state["messages"],
            ]
        )
        return {
            "messages": [
                AIMessage(
                    content=f"[WRITER]: {response.content}", name="content_writer"
                )
            ]
        }

    def content_editor(state: TeamState) -> dict:
        """Edits and polishes the writer's output."""
        response = llm.invoke(
            [
                SystemMessage(
                    content=(
                        "You are a content editor. Take the writer's draft and "
                        "improve clarity, fix any issues, and tighten the language. "
                        "Return the polished version only."
                    )
                ),
                *state["messages"],
            ]
        )
        return {
            "messages": [
                AIMessage(
                    content=f"[EDITOR]: {response.content}", name="content_editor"
                )
            ],
            "final_answer": response.content,
        }

    content_graph = StateGraph(ContentState)
    content_graph.add_node("content_supervisor", content_supervisor)
    content_graph.add_node("writer", content_writer)
    content_graph.add_node("editor", content_editor)
    content_graph.add_edge(START, "content_supervisor")
    content_graph.add_conditional_edges(
        "content_supervisor", route_content, {"writer": "writer", "editor": "editor", END: END}
    )
    # Every specialist hands control back to the supervisor
    content_graph.add_edge("writer", "content_supervisor")
    content_graph.add_edge("editor", "content_supervisor")

    return content_graph

#### 🧪 Try it: the content team and its supervisor, on their own

Run the department alone first, as with the research team. The supervisor should send the work to the writer, then the editor, then choose `FINISH`; the ✅ line checks that order in code from the specialists' message names.

In [ ]:
# ============================================================================
# TRY IT: THE CONTENT TEAM ON ITS OWN
# ============================================================================
content_result = build_content_team().compile().invoke(
    {
        "messages": [HumanMessage(content="Write a two-sentence product blurb for a reusable water bottle")],
        "final_answer": "",
    }
)

order = [m.name for m in content_result["messages"] if isinstance(m, AIMessage)]
print(f"\nSpecialists, in order: {order}")
print(f"Supervisor decisions:  {content_result['content_turns']} (cap = {MAX_CONTENT_TURNS})")
print(f"Final answer:\n{content_result['final_answer']}")

supervised = order == ["content_writer", "content_editor"] and content_result["next_step"] == "FINISH"
print(("✅" if supervised else "❌") + " the supervisor routed writer → editor, then chose FINISH")

### 2.3 📊 Analysis Team

The analysis department is also linear: a data analyst produces quantitative findings, and a strategy advisor turns those findings into actionable recommendations.

In [ ]:
# ============================================================================
# BUILD_ANALYSIS_TEAM: Analysis department: data analyst -> strategy advisor
# ============================================================================
def build_analysis_team() -> StateGraph:
    """Build the analysis department subgraph."""

    def data_analyst(state: TeamState) -> dict:
        """Provides data-driven analysis."""
        response = llm.invoke(
            [
                SystemMessage(
                    content=(
                        "You are a data analyst. Analyze the topic with numbers, "
                        "trends, and quantitative reasoning. Provide 3-4 data-driven "
                        "insights. Label every number as illustrative — do not present invented figures as real data."
                    )
                ),
                *state["messages"],
            ]
        )
        return {
            "messages": [
                AIMessage(
                    content=f"[DATA ANALYST]: {response.content}", name="data_analyst"
                )
            ]
        }

    def strategy_advisor(state: TeamState) -> dict:
        """Provides strategic recommendations."""
        response = llm.invoke(
            [
                SystemMessage(
                    content=(
                        "You are a strategy advisor. Based on the data analysis in the "
                        "conversation, provide 3 actionable strategic recommendations. "
                        "Be specific and practical."
                    )
                ),
                *state["messages"],
            ]
        )
        return {
            "messages": [
                AIMessage(
                    content=f"[STRATEGY ADVISOR]: {response.content}",
                    name="strategy_advisor",
                )
            ],
            "final_answer": response.content,
        }

    analysis_graph = StateGraph(TeamState)
    analysis_graph.add_node("data_analyst", data_analyst)
    analysis_graph.add_node("strategy_advisor", strategy_advisor)
    analysis_graph.add_edge(START, "data_analyst")
    analysis_graph.add_edge("data_analyst", "strategy_advisor")
    analysis_graph.add_edge("strategy_advisor", END)

    return analysis_graph

---
## 🎯 Part 3: Top-Level Supervisor (the CEO)

With all three departments built and compilable, the parent graph has one job: decide which department works next, hand off to that department's compiled subgraph, and take control back when it finishes. Every department edge leads back to the `ceo` node, so the CEO can send the work on to a second department (research, then content) or choose `FINISH`.

The CEO is built like the supervisor in `01_supervisor_pattern.ipynb`:
- its LLM returns **structured output** - a `DepartmentRoute` with a closed `Literal` of departments plus `FINISH` - so the decision is a typed value rather than free text to parse;
- it sees the request and the departments that already ran (read from `route_log`), and follows explicit rules: take the departments the request needs in the order it implies, never repeat one, finish when all have run;
- the decision goes to `department` and `route_log`; nothing is added to `messages`;
- `route_to_department`, a pure function, routes on `department` and stops the run after `MAX_CEO_TURNS` decisions (the `ceo_turns` counter reducer), whatever the model says.

> **Key Insight**: Each compiled department subgraph (`research_team`, `content_team`, `analysis_team`) is added to the parent graph with `add_node` exactly like a regular function node. LangGraph treats a compiled graph as just another callable, which is what makes hierarchical composition possible.

In [ ]:
# ============================================================================
# CREATE_HIERARCHICAL_SYSTEM: Parent graph: a CEO supervisor over compiled department subgraphs
# ============================================================================
CEO_PROMPT = """You are the CEO. You route each request through your departments:
- research: fact-finding, investigation, technical deep-dives
- content: writing - blog posts, marketing copy, summaries
- analysis: data analysis, strategy, business decisions

You get the request and the departments that already ran. Follow these rules exactly:
1. Work out which departments the request asks for, in the order it implies.
   "Research X, then write Y" needs research, then content. A request that asks for one thing needs one department.
2. If a needed department has not run yet, choose the first such department.
3. Never choose a department that already ran.
4. When every needed department has run, choose FINISH."""


def new_request(query: str) -> dict:
    """Initial parent state for one request: the log and the counter start empty."""
    return {
        "messages": [HumanMessage(content=query)],
        "final_answer": "",
        "department": "",
        "route_log": [],
        "ceo_turns": 0,
    }


def create_hierarchical_system():
    """
    Top-level CEO supervisor over department subgraphs.

    Each department is a compiled subgraph added as a single node, and every
    department hands control back to the CEO.
    """
    # Compile department subgraphs
    research_team = build_research_team().compile()
    content_team = build_content_team().compile()
    analysis_team = build_analysis_team().compile()

    # Supervisor routing schema
    class DepartmentRoute(BaseModel):
        department: Literal["research", "content", "analysis", "FINISH"] = Field(
            description="Which department should work next, or FINISH when every needed department has run"
        )
        reasoning: str = Field(description="Why this department was chosen")

    router_llm = llm.with_structured_output(DepartmentRoute)

    def ceo_supervisor(state: CEOState) -> dict:
        """Top-level supervisor: picks the next department, or FINISH."""
        request = next(m.content for m in state["messages"] if isinstance(m, HumanMessage))
        already_ran = [step["department"] for step in state["route_log"]]
        decision = router_llm.invoke(
            [
                SystemMessage(content=CEO_PROMPT),
                HumanMessage(
                    content=f"Request: {request}\n"
                    f"Departments that already ran, in order: {', '.join(already_ran) or 'none'}"
                ),
            ]
        )
        print(f"🏢 CEO → {decision.department}: {decision.reasoning}")
        return {
            "department": decision.department,  # the typed decision, for routing
            "route_log": [{"department": decision.department, "reasoning": decision.reasoning}],
            "ceo_turns": 1,
        }

    def route_to_department(state: CEOState) -> str:
        """Route on the CEO's typed decision stored in state, never past the cap.

        (Substring-matching the CEO's reasoning would misroute: a reason like "content needs some research"
        contains "research".)
        """
        if state["ceo_turns"] >= MAX_CEO_TURNS:
            print(f"⛔ CEO turn cap ({MAX_CEO_TURNS}) reached")
            return END
        if state["department"] == "FINISH":
            return END
        return f"{state['department']}_team"

    # Build parent graph - departments are compiled subgraphs as nodes
    parent = StateGraph(CEOState)
    parent.add_node("ceo", ceo_supervisor)
    parent.add_node("research_team", research_team)  # compiled subgraph
    parent.add_node("content_team", content_team)  # compiled subgraph
    parent.add_node("analysis_team", analysis_team)  # compiled subgraph

    parent.add_edge(START, "ceo")
    parent.add_conditional_edges(
        "ceo",
        route_to_department,
        {
            "research_team": "research_team",
            "content_team": "content_team",
            "analysis_team": "analysis_team",
            END: END,
        },
    )
    # Every department hands control back to the CEO
    parent.add_edge("research_team", "ceo")
    parent.add_edge("content_team", "ceo")
    parent.add_edge("analysis_team", "ceo")

    return parent.compile()


print("✅ create_hierarchical_system() and new_request() defined")

In [ ]:
# ============================================================================
# BUILD AND INSPECT THE HIERARCHY
# ============================================================================
system = create_hierarchical_system()
print("Parent graph:")
for edge in system.get_graph().edges:
    print(f"  {edge.source} -> {edge.target}{'  (conditional)' if edge.conditional else ''}")
print("\nWith subgraphs expanded (xray=True):")
for edge in system.get_graph(xray=True).edges:
    print(f"  {edge.source} -> {edge.target}{'  (conditional)' if edge.conditional else ''}")

---
## 🚀 Part 4: Running the Hierarchical System

These demos exercise the fully assembled hierarchy in one `.invoke()` call on the parent graph: the CEO routes a request to a department, takes control back, and either sends the work on or finishes. The `🏢 CEO` lines and the indented `📝 content supervisor` lines are printed by the two supervisors as they decide, so the output reads as a trace of both levels.

### 4.1 📋 Routing Across Multiple Queries

Three requests, each aimed at one department. For each one the CEO should pick that department, take control back, and choose `FINISH`; the ✅ line compares the CEO's decisions in `route_log` with that expected path.

In [ ]:
# ============================================================================
# DEMO_HIERARCHICAL_ROUTING: Run several queries through the full hierarchy
# ============================================================================
def demo_hierarchical_routing():
    """Run three single-department requests through the full hierarchy."""
    system = create_hierarchical_system()

    print("Hierarchical Routing Demo:\n")

    cases = [
        ("What are the latest trends in large language models?", "research"),
        ("Write a short blog introduction about AI agents", "content"),
        ("Should my startup invest in building AI features this year?", "analysis"),
    ]

    passed = 0
    for query, expected in cases:
        print(f"Query: {query}")
        print("-" * 40)
        result = system.invoke(new_request(query))

        # The CEO's decisions come from route_log, not from the transcript
        decisions = [step["department"] for step in result["route_log"]]
        ok = decisions == [expected, "FINISH"]
        passed += ok

        print(f"  Final: {result['final_answer'][:200]}...")
        print(f"  {'✅' if ok else '❌'} CEO path {' → '.join(decisions)} (expected {expected} → FINISH)")
        print("=" * 50 + "\n")

    print(f"{passed}/{len(cases)} requests went to the expected department, then FINISH")

In [ ]:
# ============================================================================
# RUN: THREE QUERIES, THREE DEPARTMENTS
# ============================================================================
demo_hierarchical_routing()

### 4.2 🔍 Full Trace of a Multi-Department Request

This request asks for two things in order: research, then writing. The CEO should send it to the research team, take control back, send it on to the content team (whose supervisor runs writer → editor on the research brief), take control back again and choose `FINISH`. The cell prints every message produced at every level, the CEO's decisions, and a ✅ computed from `route_log`.

In [ ]:
# ============================================================================
# DEMO_HIERARCHICAL_TRACE: Follow one multi-department request through both levels
# ============================================================================
def demo_hierarchical_trace():
    """Show the full trace of a request that needs two departments."""
    system = create_hierarchical_system()

    print("Full Hierarchical Trace:\n")

    result = system.invoke(
        new_request(
            "Research how retrieval-augmented generation (RAG) reduces hallucinations, "
            "then write a short blog introduction about it for developers."
        )
    )

    print("\nMessages (the work, in order):")
    for i, msg in enumerate(result["messages"]):
        if isinstance(msg, AIMessage):
            label = msg.name or "unknown"
            print(f"[Step {i}] {label}:")
            print(f"  {msg.content[:150]}...")
            print()

    decisions = [step["department"] for step in result["route_log"]]
    print(f"CEO decisions: {' → '.join(decisions)} ({result['ceo_turns']} turns, cap = {MAX_CEO_TURNS})")
    chained = decisions == ["research", "content", "FINISH"]
    print(("✅" if chained else "❌") + " the CEO chained research → content, taking control back after each, then chose FINISH")

    print(f"\nFinal answer (the content team's edited piece):\n{result['final_answer']}")

In [ ]:
# ============================================================================
# RUN: THE FULL TRACE OF ONE REQUEST
# ============================================================================
demo_hierarchical_trace()

In 4.1 each single-department request should show one department and then `FINISH`. In 4.2 the CEO chains research →
content, and the content department's own supervisor sends the draft from writer to editor: the trace shows the parallel
research specialists before the lead that merges them, then the writer drafting from that brief. Because routing reads
`department` from state, the CEO's reasoning can mention any department without changing where the request goes.

---
## 📝 Summary

### 1. Subgraphs as the unit of composition
- A department (`build_research_team`, `build_content_team`, `build_analysis_team`) is a normal `StateGraph`, compiled independently, and reused as a single node in the parent graph.
- This keeps each department testable in isolation (`demo_single_department`, the content team's own run) before it is wired into anything larger.

### 2. One shared state contract
- `TeamState` (`messages` + `final_answer`) is the contract the parent graph and every department speak, which is what lets a compiled subgraph be dropped in as a node without any adapter code.
- Each level extends it with keys only it uses: `CEOState` adds `department`, `route_log` and `ceo_turns`; `ContentState` adds `next_step` and `content_turns`. Counters stay out of `TeamState`, because a department hands its whole final state back and the parent would add a shared counter a second time.

### 3. Supervisors at two levels
- The content team's supervisor returns a `RouteDecision` (the type from `01_supervisor_pattern.ipynb`) and routes writer → editor → `FINISH`; `route_content` caps it at `MAX_CONTENT_TURNS` decisions.
- The CEO returns a typed `DepartmentRoute` and gets control back after every department, so it can chain departments (research → content) or finish. `route_to_department` reads `department` from state — no text parsing — and caps the CEO at `MAX_CEO_TURNS` decisions.
- Both supervisors write their decisions to state, never to `messages`: the transcript holds only the work.

### 4. Fan-out / fan-in inside a department
- The research team demonstrates parallel specialists (`web_researcher`, `paper_reviewer`) feeding into a synthesizing `research_lead` node - the same pattern used at any level of a hierarchy.

### Next Steps
- **Module 07 — `07_swarm_and_handoffs`**: drop the central supervisor — agents hand off to each other directly
- Compare this to the flat supervisor in `01_supervisor_pattern.ipynb` to see when a hierarchy is worth the extra structure.
- Try adding a fourth department and confirm the CEO's routing prompt and `route_to_department` logic scale cleanly.